In [ ]:
##!uv sync
##!uv add deepagents
#!pip install deepagents

In [7]:
from deepagents import create_deep_agent
import os
from google import genai
from google.genai import types
from langchain_core.tools import tool

In [8]:
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

@tool
def my_custom_tool(location: str, units: str = "metric") -> str:
    """Search for the current weather and forecast for a specified location."""

    unit_description = (
        "Celsius and kilometers per hour"
        if units == "metric"
        else "Fahrenheit and miles per hour"
    )

    prompt = f"""
Find the current weather for {location} using Google Search.

Return:
- Current temperature
- Conditions
- Feels-like temperature
- Humidity
- Wind speed
- Today's forecast

Use {unit_description}. Clearly state the location and retrieval time.
"""

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[
                types.Tool(
                    google_search=types.GoogleSearch()
                )
            ]
        ),
    )

    return response.text

In [10]:
agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[my_custom_tool],
    system_prompt="You are a research assistant. Use the weather tool for current weather.",
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the current weather in London?"
        }
    ]
})

print(result["messages"][-1].content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The current weather in London, United Kingdom, as of 07:27 AM UTC, Sunday, September 6, 2026, is 16 °C and clear. The feels-like temperature is also 16 °C. Humidity is at 68% and wind speed is 2 km/h (W), with gusts up to 3 km/h.

Today's forecast for Sunday, September 6, 2026, is mostly cloudy during the day and cloudy at night, with a 10% chance of rain throughout the day and night. Temperatures are expected to range between 14 °C and 26 °C, with humidity around 55%.


In [12]:
!pip install -U deepagents tavily-python langchain-google-genai



   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------- ----------- 0.8/1.1 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/571.7 kB ? eta -:--:--
   ---------------------------------------- 571.7/571.7 kB 6.6 MB/s  0:00:00
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   --------------------------------- ------ 786.4/941.2 kB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 941.2/941.2 kB 3.9 MB/s  0:00:00

   --- ------------------------------------  1/13 [regex]
   ------ ---------------------------------  2/13 [tiktoken]
   --------- ------------------------------  3/13 [httpcore2]
   --------- ------------------------------  3/13 [httpcore2]
   --------- -

## CREATE a Skill MD file

In [15]:
from pathlib import Path

skill_dir = Path("skills/consumer_credit_risk")
skill_dir.mkdir(parents=True, exist_ok=True)

(skill_dir / "SKILL.md").write_text(
"""# Consumer Credit Risk Analyst

## Purpose
Analyze publicly available information relevant to consumer credit risk.

## Rules
- Use `tavily_search` for current information.
- Do not use or request names, addresses, account numbers, government IDs,
  protected-class data, or other sensitive personal information.
- Do not approve, deny, price, or recommend a credit application.
- Provide analysis for qualified human review only.
- Distinguish facts, assumptions, uncertainty, and missing information.
- Cite the source title and URL for each important claim.
- Avoid inferring protected characteristics or proxies for them.

## Workflow
1. Clarify the product, geography, date range, and risk question.
2. Search for relevant economic, regulatory, market, and portfolio information.
3. Assess:
   - Macroeconomic conditions
   - Employment and income trends
   - Interest-rate environment
   - Delinquency and default trends
   - Regulatory and compliance developments
   - Model and data limitations
4. Produce a structured report:
   - Executive summary
   - Evidence and sources
   - Risk indicators
   - Scenario analysis
   - Limitations
   - Human-review considerations

## Output requirement
Clearly state that the report is analytical and must not be used as the sole
basis for an individual credit decision.
""",
encoding="utf-8",
)

1319

### Define the Tavily search tool:

In [ ]:
// %pip install -U python-dotenv
import os
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    raise RuntimeError("TAVILY_API_KEY was not found in the .env file.")

tavily_client = TavilyClient(api_key=tavily_api_key)

In [21]:
import os
from tavily import TavilyClient
from langchain_core.tools import tool

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def tavily_search(query: str) -> str:
    """Search the web for current consumer credit risk information."""
    response = tavily_client.search(
        query=query,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )

    results = []
    if response.get("answer"):
        results.append(f"Summary:\n{response['answer']}")

    for item in response.get("results", []):
        results.append(
            f"Title: {item.get('title', '')}\n"
            f"URL: {item.get('url', '')}\n"
            f"Content: {item.get('content', '')}"
        )

    return "\n\n".join(results) or "No search results found."

In [22]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[tavily_search],
    skills=[str(skill_dir.resolve())],
    system_prompt=(
        "You are a consumer credit risk analyst. "
        "Use the consumer_credit_risk skill and Tavily search. "
        "Never make individual lending decisions."
    ),
)

### tavily_search as a tool;  

In [23]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Analyze current US consumer credit risk conditions for unsecured "
            "personal loans. Cover delinquency trends, interest rates, "
            "employment, regulatory concerns, and downside scenarios."
        )
    }]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': '## Analysis of Current US Consumer Credit Risk Conditions for Unsecured Personal Loans\n\nThe US consumer credit risk landscape for unsecured personal loans presents a nuanced picture, characterized by both resilience and emerging areas of concern. While overall credit conditions remain stable, specific segments of the market, particularly lower-credit-tier borrowers, are experiencing increased pressure.\n\n### Delinquency Trends\n\nDelinquency rates for unsecured personal loans show mixed signals. TransUnion data indicates a slight uptick in borrower-level delinquency rates (60+ days past due), rising from 3.49% in Q1 2025 to 3.98% in Q1 2026, and from 3.37% in Q2 2025 to 3.81% in Q2 2026. This increase is attributed to lenders reaching more consumers, especially at the subprime end, while managing risk through smaller balances and tighter underwriting.\n\nConversely, VantageScore\'s May 2026 CreditGauge™ reports an improvement in early-stage delinquencies (